# 01 — Exploratory Data Analysis
## PowerGuard AI

**Purpose:** Understand the structure, distributions, patterns, and quality of both processed datasets before building any ML model.

**Datasets used (read-only, from `data/processed/`):**
- `transformer_features.csv` — 470 transformer oil/DGA samples, 20 features
- `transformer_target.csv` — 470 rows, risk_class labels
- `weather_outage_events.csv` — 33,139 outage events, weather features + severity target

**No raw files are modified. No models are trained.**

---
### Table of Contents
1. Setup & Data Loading
2. Dataset A — Transformer Health (DGA)
   - 2.1 Basic Profile
   - 2.2 Target Variable: Risk Class Distribution
   - 2.3 DGA Gas Feature Distributions
   - 2.4 Electrical & Oil Quality Features
   - 2.5 Engineered Ratio Features
   - 2.6 Feature Distributions by Risk Class
   - 2.7 Outlier Detection
   - 2.8 Correlation Analysis
3. Dataset B — Weather + Outage Events
   - 3.1 Basic Profile
   - 3.2 Target Variable: Outage Severity Distribution
   - 3.3 Outage Magnitude Distribution
   - 3.4 Weather Feature Distributions
   - 3.5 Time Patterns
   - 3.6 Location Patterns
   - 3.7 Weather vs Outage Severity
   - 3.8 Missing Values
4. Cross-Dataset Architecture Summary
5. EDA Summary & ML Recommendations

## 1. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# ── Consistent style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':        110,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
})

RISK_COLORS    = {'HIGH': '#dc2626', 'MEDIUM': '#d97706', 'LOW': '#16a34a'}
RISK_ORDER     = ['HIGH', 'MEDIUM', 'LOW']
SEVERITY_ORDER = ['HIGH', 'MEDIUM', 'LOW']
SEV_COLORS     = {'HIGH': '#7c3aed', 'MEDIUM': '#2563eb', 'LOW': '#64748b'}

# ── Paths ─────────────────────────────────────────────────────────────────────
PROC = Path('..') / 'data' / 'processed'

# ── Load ──────────────────────────────────────────────────────────────────────
feat_df   = pd.read_csv(PROC / 'transformer_features.csv')
target_df = pd.read_csv(PROC / 'transformer_target.csv')
wx_df     = pd.read_csv(PROC / 'weather_outage_events.csv', parse_dates=['event_start'])

# Merged transformer dataset (features + target)
dga = feat_df.merge(target_df, on='sample_id').copy()

print(f'Transformer dataset : {dga.shape[0]:,} rows  x  {dga.shape[1]} cols')
print(f'Weather/outage events: {wx_df.shape[0]:,} rows  x  {wx_df.shape[1]} cols')

## 2. Dataset A — Transformer Health (DGA)

### 2.1 Basic Profile

In [ ]:
print('Shape:', dga.shape)
print('\nDtypes:')
print(dga.dtypes.to_string())
print('\nMissing values:', dga.isnull().sum().sum())
print('\nDescriptive statistics:')
dga.describe().T.round(3)

### 2.2 Target Variable: Risk Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Panel 1: Risk class counts ────────────────────────────────────────────────
ax = axes[0]
counts = dga['risk_class'].value_counts().reindex(RISK_ORDER)
bars = ax.bar(counts.index, counts.values,
               color=[RISK_COLORS[c] for c in counts.index], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(val), ha='center', va='bottom', fontweight='bold')
ax.set_title('Risk Class Distribution')
ax.set_ylabel('Count')
ax.set_xlabel('Risk Class')

# ── Panel 2: Pie chart ────────────────────────────────────────────────────────
ax = axes[1]
pcts = counts / counts.sum() * 100
wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=[f'{c}\n({v:.1f}%)' for c, v in zip(counts.index, pcts)],
    colors=[RISK_COLORS[c] for c in counts.index],
    autopct='',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax.set_title('Risk Class Proportions')

# ── Panel 3: Health index distribution by class ───────────────────────────────
ax = axes[2]
for cls in RISK_ORDER:
    subset = dga.loc[dga['risk_class'] == cls, 'health_index']
    ax.hist(subset, bins=15, alpha=0.6, color=RISK_COLORS[cls], label=cls, edgecolor='white')
ax.axvline(25, color='#dc2626', linestyle='--', linewidth=1, label='HIGH/MEDIUM boundary (25)')
ax.axvline(50, color='#16a34a', linestyle='--', linewidth=1, label='MEDIUM/LOW boundary (50)')
ax.set_title('Health Index Distribution by Risk Class')
ax.set_xlabel('Health Index Score (0=worst, 100=best)')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

plt.suptitle('Dataset A — Risk Target Overview', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\nClass imbalance note: 57% HIGH, 32% MEDIUM, 11% LOW.')
print('Model training MUST use class_weight="balanced" or oversampling.')

### 2.3 DGA Gas Feature Distributions

In [ ]:
dga_gas_cols = [
    'hydrogen_ppm', 'oxygen_ppm', 'nitrogen_ppm', 'methane_ppm',
    'co_ppm', 'co2_ppm', 'ethylene_ppm', 'ethane_ppm', 'acetylene_ppm'
]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, col in enumerate(dga_gas_cols):
    ax = axes[i]
    data = dga[col]

    # Log1p transform because gas concentrations are highly right-skewed
    ax.hist(np.log1p(data), bins=40, color='#3b82d4', alpha=0.75, edgecolor='white', linewidth=0.4)

    zero_pct = (data == 0).mean() * 100
    ax.set_title(f'{col}\n(zeros: {zero_pct:.0f}%)', fontsize=10)
    ax.set_xlabel('log1p(ppm)', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)

    # Annotate median
    med = data.median()
    ax.axvline(np.log1p(med), color='#dc2626', linestyle='--', linewidth=1.2,
               label=f'median={med:.0f}')
    ax.legend(fontsize=8)

plt.suptitle('DGA Gas Concentrations — log1p distributions (all 470 samples)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('\nKey observations:')
print('- hydrogen, methane, ethylene, ethane: bimodal — low values in poor-health transformers')
print('- acetylene: 92% zeros (arcing faults are rare; presence is a strong fault indicator)')
print('- nitrogen, oxygen: background gases — present in all samples, high variance')
print('- All distributions are heavily right-skewed (log1p helps reveal structure)')

### 2.4 Electrical & Oil Quality Features

In [ ]:
oil_cols = [
    'dbds_mg_kg', 'power_factor_pct', 'interfacial_tension_mNm',
    'dielectric_kv', 'water_content_ppm'
]
oil_labels = {
    'dbds_mg_kg':             'DBDS (mg/kg)\nCorrosive sulphur',
    'power_factor_pct':       'Power Factor (%)\nInsulation quality',
    'interfacial_tension_mNm':'Interfacial Tension (mN/m)\nOil oxidation',
    'dielectric_kv':          'Dielectric Rigidity (kV)\nBreakdown voltage',
    'water_content_ppm':      'Water Content (ppm)\nMoisture ingress',
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(oil_cols):
    ax = axes[i]
    for cls in RISK_ORDER:
        subset = dga.loc[dga['risk_class'] == cls, col]
        ax.hist(subset, bins=25, alpha=0.55, color=RISK_COLORS[cls],
                label=cls, edgecolor='white', linewidth=0.3)
    ax.set_title(oil_labels[col], fontsize=10)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

# Hide the unused 6th panel
axes[5].set_visible(False)

plt.suptitle('Electrical & Oil Quality Features by Risk Class', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('\nKey observations:')
print('- power_factor_pct: HIGH-risk transformers cluster at LOW values — counter-intuitive!')
print('  Explanation: Most HIGH-risk samples come from very degraded units where oil is')
print('  too contaminated to give a valid reading (stuck at 1.0).')
print('- interfacial_tension_mNm: LOW-risk units have HIGHER IFT — expected (fresh oil).')
print('- dielectric_kv: wider spread in LOW-risk, HIGH-risk cluster around 50-55 kV.')
print('- water_content_ppm: HIGH-risk units have the HIGHEST moisture — strong failure signal.')
print('- dbds_mg_kg: mostly zero for HIGH-risk; non-zero concentrated in MEDIUM/LOW.')

### 2.5 Engineered Ratio Features

In [ ]:
ratio_cols = [
    'tdcg_ppm', 'rogers_r1_ch4_h2', 'rogers_r2_c2h2_c2h4',
    'rogers_r3_c2h2_ch4', 'co2_co_ratio', 'ethylene_ethane_ratio'
]
ratio_desc = {
    'tdcg_ppm':               'TDCG (ppm)\nTotal combustible gas',
    'rogers_r1_ch4_h2':       'Rogers R1: CH4/H2\nThermal vs partial discharge',
    'rogers_r2_c2h2_c2h4':    'Rogers R2: C2H2/C2H4\nArcing fault indicator',
    'rogers_r3_c2h2_ch4':     'Rogers R3: C2H2/CH4\nDischarge type',
    'co2_co_ratio':           'CO2/CO Ratio\nCellulose degradation rate',
    'ethylene_ethane_ratio':  'C2H4/C2H6 Ratio\nHigh-temp thermal fault',
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(ratio_cols):
    ax = axes[i]
    data_log = np.log1p(dga[col])
    for cls in RISK_ORDER:
        subset = np.log1p(dga.loc[dga['risk_class'] == cls, col])
        ax.hist(subset, bins=30, alpha=0.55, color=RISK_COLORS[cls],
                label=cls, edgecolor='white', linewidth=0.3)
    ax.set_title(ratio_desc[col], fontsize=10)
    ax.set_xlabel('log1p(value)', fontsize=9)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Engineered IEC Ratio Features by Risk Class (log1p scale)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('\nKey observations:')
print('- TDCG: LOW-risk transformers have MUCH higher TDCG (~7,189 mean vs ~298 for HIGH).')
print('  This seems reversed — it reflects the dataset skew: "LOW risk" here means')
print('  health_index >= 50, and many of those are older units with moderate gas buildup.')
print('- Rogers R2 (C2H2/C2H4): discriminates arcing; mostly near-zero across all classes.')
print('- CO2/CO ratio: overlapping distributions — less discriminative on its own.')

### 2.6 Feature Distributions by Risk Class — Box Plots

In [ ]:
# Select features most likely to discriminate by risk class
box_features = [
    'hydrogen_ppm', 'acetylene_ppm', 'water_content_ppm',
    'power_factor_pct', 'interfacial_tension_mNm', 'tdcg_ppm'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(box_features):
    ax = axes[i]
    grouped = [np.log1p(dga.loc[dga['risk_class'] == cls, col].values) for cls in RISK_ORDER]
    bp = ax.boxplot(grouped, patch_artist=True, notch=False,
                    medianprops={'color': 'black', 'linewidth': 2},
                    whiskerprops={'linewidth': 1.2},
                    flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4})
    for patch, cls in zip(bp['boxes'], RISK_ORDER):
        patch.set_facecolor(RISK_COLORS[cls])
        patch.set_alpha(0.7)
    ax.set_xticklabels(RISK_ORDER)
    ax.set_title(f'log1p({col})', fontsize=10)
    ax.set_xlabel('Risk Class')
    ax.set_ylabel('log1p(value)')

plt.suptitle('Key Feature Distributions by Risk Class (log1p scale)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Reading guide:')
print('- hydrogen_ppm: LOW-risk >> MEDIUM >= HIGH — degraded units have depleted H2')
print('- acetylene_ppm: LOW-risk > others — trace arcing in aging units')
print('- water_content_ppm: HIGH-risk >> others — moisture is the best single predictor')
print('- power_factor_pct: LOW-risk has higher spread — degradation causes extreme readings')
print('- interfacial_tension: LOW-risk has higher IFT — fresher oil')
print('- tdcg_ppm: LOW-risk >> HIGH-risk — see dataset skew note above')

### 2.7 Outlier Detection

In [ ]:
raw_sensor_cols = [
    'hydrogen_ppm', 'oxygen_ppm', 'nitrogen_ppm', 'methane_ppm',
    'co_ppm', 'co2_ppm', 'ethylene_ppm', 'ethane_ppm', 'acetylene_ppm',
    'dbds_mg_kg', 'power_factor_pct', 'interfacial_tension_mNm',
    'dielectric_kv', 'water_content_ppm'
]

print('IQR-based outlier count per feature (values beyond Q1 - 1.5*IQR or Q3 + 1.5*IQR):')
print(f'{"Feature":<35} {"Outliers":>8} {"% of total":>10} {"Max value":>12}')
print('-' * 70)
for col in raw_sensor_cols:
    q1, q3 = dga[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((dga[col] < lo) | (dga[col] > hi)).sum()
    pct = n_out / len(dga) * 100
    print(f'{col:<35} {n_out:>8} {pct:>9.1f}% {dga[col].max():>12.1f}')

print('\nNote: oxygen_ppm was already capped to 99th percentile in the pipeline.')
print('High outlier rates are expected in DGA data — fault gases follow power-law distributions.')
print('These are NOT data errors; they represent real transformer faults.')

In [ ]:
# Scatter plot: two strongest predictors vs health_index
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

scatter_pairs = [
    ('water_content_ppm', 'health_index', 'Water Content vs Health Index'),
    ('hydrogen_ppm',      'health_index', 'Hydrogen (log) vs Health Index'),
    ('dbds_mg_kg',        'health_index', 'DBDS vs Health Index'),
]

for ax, (xcol, ycol, title) in zip(axes, scatter_pairs):
    for cls in RISK_ORDER:
        mask = dga['risk_class'] == cls
        xvals = np.log1p(dga.loc[mask, xcol]) if xcol != 'water_content_ppm' else dga.loc[mask, xcol]
        ax.scatter(xvals, dga.loc[mask, ycol],
                   c=RISK_COLORS[cls], alpha=0.45, s=20, label=cls)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('log1p(value)' if xcol not in ('water_content_ppm', 'dbds_mg_kg') else xcol, fontsize=9)
    ax.set_ylabel('Health Index', fontsize=9)
    ax.axhline(25, color='#dc2626', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.axhline(50, color='#16a34a', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.legend(fontsize=8)

plt.suptitle('Feature vs Health Index — Key Relationships', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 2.8 Correlation Analysis

In [ ]:
# Correlation matrix — features + health_index (to see structure)
corr_cols = raw_sensor_cols + ['health_index', 'risk_int']
corr_matrix = dga[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.7)

labels = corr_cols
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(labels, fontsize=8)

# Annotate cells
for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr_matrix.values[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6.5, color=color)

ax.set_title('Correlation Matrix — DGA Features + Target', fontsize=13)
plt.tight_layout()
plt.show()

# Top correlations with health_index and risk_int
print('Top correlations with health_index (descending |r|):')
corr_hi = corr_matrix['health_index'].drop(['health_index', 'risk_int']).abs().sort_values(ascending=False)
for feat, val in corr_hi.items():
    sign = corr_matrix.loc[feat, 'health_index']
    print(f'  {feat:<35} r = {sign:+.3f}')

## 3. Dataset B — Weather + Outage Events

### 3.1 Basic Profile

In [ ]:
print('Shape:', wx_df.shape)
print('\nDtypes:')
print(wx_df.dtypes.to_string())
print('\nMissing values per column:', wx_df.isnull().sum().to_dict())
print('\nDescriptive statistics:')
wx_df.describe().T.round(3)

### 3.2 Target Variable: Outage Severity Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Panel 1: Severity counts ──────────────────────────────────────────────────
ax = axes[0]
counts = wx_df['outage_severity'].value_counts().reindex(SEVERITY_ORDER)
bars = ax.bar(counts.index, counts.values,
               color=[SEV_COLORS[c] for c in counts.index], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_title('Outage Severity Distribution')
ax.set_ylabel('Number of Events')

# ── Panel 2: Pie ──────────────────────────────────────────────────────────────
ax = axes[1]
pcts = counts / counts.sum() * 100
ax.pie(
    counts.values,
    labels=[f'{c}\n({v:.1f}%)' for c, v in zip(counts.index, pcts)],
    colors=[SEV_COLORS[c] for c in counts.index],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax.set_title('Severity Proportions')

# ── Panel 3: Customers affected log-distribution ──────────────────────────────
ax = axes[2]
for cls in SEVERITY_ORDER:
    subset = wx_df.loc[wx_df['outage_severity'] == cls, 'max_customers_affected']
    ax.hist(np.log1p(subset), bins=40, alpha=0.6,
            color=SEV_COLORS[cls], label=f'{cls} (n={len(subset):,})', edgecolor='white')
ax.set_title('Max Customers Affected\n(log1p scale)')
ax.set_xlabel('log1p(customers)')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

plt.suptitle('Dataset B — Outage Severity Target Overview', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\nSevere class imbalance: only 1.4% of events are HIGH severity.')
print('Binning thresholds: LOW < 10 customers, MEDIUM 10-1000, HIGH > 1000.')
print('MEDIUM and HIGH combined = 36.7% of events but represent almost all grid impact.')

### 3.3 Outage Magnitude Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: Raw distribution
ax = axes[0]
ax.hist(wx_df['max_customers_affected'], bins=100, color='#3b82d4', alpha=0.75, edgecolor='white')
ax.set_title('Max Customers Affected (raw)')
ax.set_xlabel('Customers')
ax.set_ylabel('Count')
ax.set_xlim(0, 5000)  # cap display at 5000 to see main body

# Panel 2: Log1p distribution
ax = axes[1]
ax.hist(np.log1p(wx_df['max_customers_affected']), bins=60,
        color='#3b82d4', alpha=0.75, edgecolor='white')
ax.axvline(np.log1p(10), color='#d97706', linestyle='--', linewidth=1.5, label='LOW/MED boundary (10)')
ax.axvline(np.log1p(1000), color='#7c3aed', linestyle='--', linewidth=1.5, label='MED/HIGH boundary (1000)')
ax.set_title('Max Customers Affected (log1p)')
ax.set_xlabel('log1p(customers)')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# Panel 3: CDF
ax = axes[2]
sorted_vals = np.sort(wx_df['max_customers_affected'])
cdf = np.arange(1, len(sorted_vals)+1) / len(sorted_vals)
ax.plot(np.log1p(sorted_vals), cdf, color='#2563eb', linewidth=1.5)
ax.axvline(np.log1p(10), color='#d97706', linestyle='--', linewidth=1.2, label='10 customers')
ax.axvline(np.log1p(1000), color='#7c3aed', linestyle='--', linewidth=1.2, label='1000 customers')
ax.set_title('Cumulative Distribution of Outage Size')
ax.set_xlabel('log1p(customers affected)')
ax.set_ylabel('Cumulative probability')
ax.legend(fontsize=9)

plt.suptitle('Outage Magnitude Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

pcts = wx_df['outage_severity'].value_counts(normalize=True).reindex(SEVERITY_ORDER) * 100
print('\nOutage size percentile summary:')
for p in [25, 50, 75, 90, 95, 99]:
    v = wx_df['max_customers_affected'].quantile(p/100)
    print(f'  P{p:02d}: {v:.1f} customers')

### 3.4 Weather Feature Distributions

In [ ]:
weather_cols = ['tmpf', 'relh', 'sknt', 'p01i']
weather_labels = {
    'tmpf': 'Temperature (°F)',
    'relh': 'Relative Humidity (%)',
    'sknt': 'Wind Speed (knots)',
    'p01i': 'Precipitation (in/hr)',
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, col in enumerate(weather_cols):
    # Top row: full distribution
    ax = axes[0, i]
    ax.hist(wx_df[col], bins=50, color='#3b82d4', alpha=0.75, edgecolor='white', linewidth=0.3)
    ax.set_title(weather_labels[col], fontsize=10)
    ax.set_ylabel('Count')
    med = wx_df[col].median()
    ax.axvline(med, color='#dc2626', linestyle='--', linewidth=1.5, label=f'median={med:.1f}')
    ax.legend(fontsize=8)

    # Bottom row: by severity class
    ax = axes[1, i]
    for sev in SEVERITY_ORDER:
        subset = wx_df.loc[wx_df['outage_severity'] == sev, col]
        ax.hist(subset, bins=40, alpha=0.55,
                color=SEV_COLORS[sev], label=sev, edgecolor='white', linewidth=0.2)
    ax.set_title(f'{weather_labels[col]}\nby Severity', fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Weather Feature Distributions (top: all events, bottom: by severity)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('\nMean weather values by severity:')
print(wx_df.groupby('outage_severity')[weather_cols].mean().reindex(SEVERITY_ORDER).round(2).to_string())

### 3.5 Time Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ── Panel 1: Events by month ──────────────────────────────────────────────────
ax = axes[0, 0]
month_counts = wx_df.groupby(['month', 'outage_severity']).size().unstack(fill_value=0)
month_names = {4:'Apr', 5:'May', 6:'Jun', 7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct'}
x = range(len(month_counts))
bottom = np.zeros(len(month_counts))
for sev in SEVERITY_ORDER:
    if sev in month_counts.columns:
        vals = month_counts[sev].values
        ax.bar(x, vals, bottom=bottom, color=SEV_COLORS[sev], label=sev, alpha=0.85)
        bottom += vals
ax.set_xticks(list(x))
ax.set_xticklabels([month_names.get(m, str(m)) for m in month_counts.index])
ax.set_title('Outage Events by Month')
ax.set_ylabel('Number of Events')
ax.legend()

# ── Panel 2: Events by year ───────────────────────────────────────────────────
ax = axes[0, 1]
year_counts = wx_df.groupby(['year', 'outage_severity']).size().unstack(fill_value=0)
x = range(len(year_counts))
bottom = np.zeros(len(year_counts))
for sev in SEVERITY_ORDER:
    if sev in year_counts.columns:
        vals = year_counts[sev].values
        ax.bar(x, vals, bottom=bottom, color=SEV_COLORS[sev], label=sev, alpha=0.85)
        bottom += vals
ax.set_xticks(list(x))
ax.set_xticklabels(year_counts.index, rotation=30)
ax.set_title('Outage Events by Year')
ax.set_ylabel('Number of Events')
ax.legend()

# ── Panel 3: Events by hour of day ────────────────────────────────────────────
ax = axes[1, 0]
hour_counts = wx_df.groupby(['hour_of_day', 'outage_severity']).size().unstack(fill_value=0)
x = range(24)
bottom = np.zeros(len(hour_counts))
for sev in SEVERITY_ORDER:
    if sev in hour_counts.columns:
        vals = hour_counts.reindex(range(24), fill_value=0)[sev].values
        ax.bar(x, vals, bottom=bottom, color=SEV_COLORS[sev], label=sev, alpha=0.85, width=0.8)
        bottom += vals
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 3)])
ax.set_title('Outage Events by Hour of Day')
ax.set_ylabel('Number of Events')
ax.set_xlabel('Hour (UTC)')
ax.legend()

# ── Panel 4: HIGH severity events by year ─────────────────────────────────────
ax = axes[1, 1]
high_by_year = wx_df[wx_df['outage_severity'] == 'HIGH'].groupby('year').size()
ax.plot(high_by_year.index, high_by_year.values, marker='o', color='#7c3aed',
        linewidth=2, markersize=8)
ax.fill_between(high_by_year.index, high_by_year.values, alpha=0.15, color='#7c3aed')
ax.set_title('HIGH Severity Events per Year')
ax.set_ylabel('Count')
ax.set_xlabel('Year')

plt.suptitle('Temporal Patterns in Outage Events', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('Key temporal patterns:')
print('- 95% of events occur in SUMMER months (Jun-Aug) — heat-driven outages dominate')
print('- July alone accounts for 58% of all events')
print('- Outage counts grow year-over-year from 2015 to 2022 (possible dataset expansion)')
print('- Evening hours (18:00-23:00) have slightly more events — peak load + storm timing')
print('- Morning hours (05:00-08:00) have fewest events — overnight cool-down, low load')

### 3.6 Location Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel 1: Top 15 states by event count ─────────────────────────────────────
ax = axes[0]
top_states = wx_df['state'].value_counts().head(15)
ax.barh(top_states.index[::-1], top_states.values[::-1], color='#3b82d4', alpha=0.8, edgecolor='white')
ax.set_title('Top 15 States by Outage Event Count')
ax.set_xlabel('Number of Events')

# ── Panel 2: HIGH severity events by state ────────────────────────────────────
ax = axes[1]
high_states = wx_df[wx_df['outage_severity'] == 'HIGH']['state'].value_counts().head(15)
ax.barh(high_states.index[::-1], high_states.values[::-1], color='#7c3aed', alpha=0.8, edgecolor='white')
ax.set_title('Top 15 States — HIGH Severity Events Only')
ax.set_xlabel('Number of HIGH Events')

plt.suptitle('Geographic Distribution of Outages', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# State HIGH-severity rate
state_total = wx_df.groupby('state').size()
state_high  = wx_df[wx_df['outage_severity']=='HIGH'].groupby('state').size()
state_rate  = (state_high / state_total * 100).sort_values(ascending=False).dropna()
print('\nTop 10 states by HIGH-severity rate (% of that state events):')
print(state_rate.head(10).round(1).to_string())
print('\nNote: California dominates HIGH-severity events (192/456 = 42%).')
print('Texas and Mississippi are top by volume but not by HIGH-severity rate.')

### 3.7 Weather vs Outage Severity

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (col, label) in enumerate(zip(weather_cols, weather_labels.values())):
    ax = axes[i // 2, i % 2]
    grouped = [wx_df.loc[wx_df['outage_severity'] == sev, col].values for sev in SEVERITY_ORDER]
    bp = ax.boxplot(grouped, patch_artist=True, notch=False,
                    medianprops={'color': 'black', 'linewidth': 2},
                    flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.3})
    for patch, sev in zip(bp['boxes'], SEVERITY_ORDER):
        patch.set_facecolor(SEV_COLORS[sev])
        patch.set_alpha(0.75)
    ax.set_xticklabels(SEVERITY_ORDER)
    ax.set_title(label)
    ax.set_xlabel('Outage Severity')

plt.suptitle('Weather Features by Outage Severity', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Correlation
print('Pearson correlation with outage_severity_int:')
for col in weather_cols:
    r = wx_df[col].corr(wx_df['outage_severity_int'])
    print(f'  {col:<20} r = {r:+.4f}')

print()
print('Key observations:')
print('- relh (humidity): negative correlation — HIGH outages cluster at LOWER humidity.')
print('  This is counter-intuitive but consistent: California wildfires/heatwaves cause')
print('  high outages in dry conditions (low humidity).')
print('- tmpf: near-zero correlation overall; HIGH events slightly cooler on average.')
print('- sknt, p01i: small positive correlations — wind and rain contribute to outages.')
print('- Individual weather features have weak linear signal (max |r| ~ 0.14).')
print('  Non-linear models (RF) or interaction features will be needed.')

### 3.8 Missing Values (Before Imputation — Original Distribution)

In [ ]:
# The processed dataset has zero missing values (imputed in pipeline).
# Reconstruct the original missingness picture from the raw data.
raw_weather = pd.read_csv(
    Path('..') / 'data' / 'raw' / 'Weather_data_combined_with_outage.csv',
    na_values=['NA', 'N/A', '']
)

print('Raw Weather file — missing value summary:')
missing = raw_weather.isnull().sum()
missing_pct = (missing / len(raw_weather) * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
miss_data = missing_pct[missing_pct > 0].sort_values(ascending=False)
colors = ['#dc2626' if v > 50 else '#d97706' if v > 10 else '#64748b' for v in miss_data.values]
bars = ax.bar(miss_data.index, miss_data.values, color=colors, edgecolor='white')
for bar, val in zip(bars, miss_data.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Missing Value Rate in Raw Weather Dataset')
ax.set_ylabel('% Missing')
ax.set_xlabel('Column')
ax.axhline(50, color='#dc2626', linestyle='--', linewidth=1, alpha=0.7, label='50% threshold')
ax.axhline(10, color='#d97706', linestyle='--', linewidth=1, alpha=0.7, label='10% threshold')
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

for col in miss_data.index:
    action = 'DROPPED' if miss_data[col] > 50 else 'IMPUTED with median' if miss_data[col] > 1 else 'Imputed with 0'
    print(f'  {col:<25} {miss_data[col]:>5.1f}% missing  ->  {action}')

## 4. Cross-Dataset Architecture Summary

In [ ]:
print('='*60)
print('CROSS-DATASET SUMMARY')
print('='*60)
print()
print('Dataset A: transformer_features.csv + transformer_target.csv')
print(f'  Samples : 470')
print(f'  Features: 20 (14 raw DGA sensors + 6 IEC ratio features)')
print(f'  Target  : risk_class (HIGH=57%, MEDIUM=32%, LOW=11%)')
print(f'  Missing : 0')
print(f'  Grain   : one oil sample per transformer')
print()
print('Dataset B: weather_outage_events.csv')
print(f'  Events  : 33,139')
print(f'  Features: 4 weather + 4 temporal + 3 location = 11 usable features')
print(f'  Target  : outage_severity (HIGH=1.4%, MEDIUM=35%, LOW=63%)')
print(f'  Missing : 0 (imputed)')
print(f'  Grain   : one outage event per county')
print()
print('Join: NOT POSSIBLE (no shared key)')
print('Integration: Score fusion at inference time')
print()
print('Neither dataset contains:')
print('  - Vibration data (not in these datasets)')
print('  - Voltage / current / power readings (not in these datasets)')
print('  - Real-time sensor streams')
print('  - Asset IDs / transformer serial numbers')
print('  These would need to be added from a SCADA/asset management system.')

## 5. EDA Summary & ML Recommendations

In [ ]:
# ── Final summary visualisation: feature importance proxy ────────────────────
# Use absolute Spearman correlation with target (handles non-linearity better)
from scipy.stats import spearmanr

feat_cols = [
    'hydrogen_ppm', 'oxygen_ppm', 'nitrogen_ppm', 'methane_ppm',
    'co_ppm', 'co2_ppm', 'ethylene_ppm', 'ethane_ppm', 'acetylene_ppm',
    'dbds_mg_kg', 'power_factor_pct', 'interfacial_tension_mNm',
    'dielectric_kv', 'water_content_ppm',
    'tdcg_ppm', 'rogers_r1_ch4_h2', 'rogers_r2_c2h2_c2h4',
    'rogers_r3_c2h2_ch4', 'co2_co_ratio', 'ethylene_ethane_ratio'
]

spearman_corrs = {}
for col in feat_cols:
    r, _ = spearmanr(dga[col], dga['risk_int'])
    spearman_corrs[col] = abs(r)

sc = pd.Series(spearman_corrs).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#dc2626' if v > 0.3 else '#d97706' if v > 0.15 else '#64748b' for v in sc.values]
ax.barh(sc.index, sc.values, color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0.3, color='#dc2626', linestyle='--', linewidth=1.2, alpha=0.7, label='|rho| > 0.3 (strong)')
ax.axvline(0.15, color='#d97706', linestyle='--', linewidth=1.2, alpha=0.7, label='|rho| > 0.15 (moderate)')
ax.set_title('Spearman |rho| with risk_int — Feature Predictive Signal', fontsize=13)
ax.set_xlabel('|Spearman correlation with risk_int|')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 10 features by Spearman |rho| with risk_int:')
for feat, val in sc.sort_values(ascending=False).head(10).items():
    print(f'  {feat:<35} |rho| = {val:.4f}')

In [ ]:
print('='*70)
print('EDA SUMMARY')
print('='*70)

print('''
1. MOST IMPORTANT PATTERNS
---
Dataset A (Transformer DGA):
  - water_content_ppm is the STRONGEST single predictor (high moisture = HIGH risk).
  - hydrogen_ppm is high in LOW-risk (healthy) transformers; depleted in HIGH-risk.
    This is because HIGH-risk units in this dataset are mostly OLD, DEGRADED units
    with exhausted fault gas capacity, not newly faulting ones.
  - oxygen_ppm had 65 outliers (up to 249,900 ppm) — capped to 99th percentile.
  - acetylene_ppm is 92% zero — when non-zero, it strongly indicates arcing.
  - The dataset is SKEWED: 89% of samples have Health Index < 50 (unhealthy units).
  - TDCG appears higher in LOW-risk — this is a consequence of dataset skew, not
    a physical contradiction (older "borderline" units still producing gases).

Dataset B (Weather + Outage):
  - 95% of events occur in SUMMER (June–August). Month/season is a key feature.
  - July accounts for 58% of all outage events.
  - California dominates HIGH-severity events (42% of all HIGH events).
  - HIGH-severity events have LOWER humidity — driven by dry-condition wildfires/heatwaves.
  - Individual weather features have weak linear correlation (max Pearson r ~ 0.14).
  - Outage magnitude spans 4 orders of magnitude (0.06 to 62,374 customers).
  - 63% of events are LOW severity (< 10 customers) — class imbalance is severe.
  - Outage counts increase year-over-year (2015→2022), suggesting dataset growth.
''')

print('''
2. FEATURES POTENTIALLY RELATED TO FAILURE
---
Dataset A (DGA transformer health):
  STRONG signal (Spearman |rho| > 0.30 with risk_int):
    - water_content_ppm     : moisture is the primary aging/failure driver
    - interfacial_tension_mNm: oil oxidation indicator
    - ethylene_ppm / ethylene_ethane_ratio: high-temp thermal fault indicators
    - hydrogen_ppm          : major fault gas
    - rogers_r3_c2h2_ch4    : discharge/thermal discrimination ratio

  MODERATE signal (|rho| 0.15-0.30):
    - methane_ppm, ethane_ppm
    - acetylene_ppm (rare but critical when present)
    - tdcg_ppm (total fault gas burden)
    - dbds_mg_kg (corrosive sulphur)
    - rogers_r1_ch4_h2

Dataset B (weather/outage):
  - relh (humidity): negative correlation — low humidity => high severity
  - month + season: summer dominates all severity levels
  - p01i (precipitation): positive but weak
  - sknt (wind speed): positive but weak
  - state/fips_code: strong geographic clustering (California, Texas)
''')

print('''
3. FEATURES THAT SHOULD PROBABLY BE REMOVED
---
Dataset A:
  - health_index:          TARGET — already excluded from features file
  - life_expectation_years:DERIVED from target — excluded from features file
  - nitrogen_ppm:          background gas, near-zero signal with target
  - co2_co_ratio:          weak linear signal; CO and CO2 individually carry more
  - oxygen_ppm:            background gas; weak signal after capping outlier

Dataset B (already dropped in pipeline):
  - max_rolling_avg, evDur_Hr, evDur_day: DATA LEAKAGE (already removed)
  - median_sum: SOFT LEAKAGE (already removed)
  - gust: 91.5% missing (already removed)
  - feel: collinear with tmpf + relh (already removed)
  - state_st: redundant (already removed)

To consider removing at modelling time:
  - event_id: identifier, no predictive value
  - fips_code: high-cardinality ID — encode as state or use state only
  - county: 924 unique values — too high cardinality for naive encoding
''')

print('''
4. POTENTIAL DATA LEAKAGE
---
Dataset A:
  - health_index and life_expectation_years: DIRECTLY derived from features.
    MITIGATED: written to a separate target CSV file.
  - Engineered ratio features (rogers ratios, tdcg, co2_co_ratio):
    Computed from raw features only — NO leakage.

Dataset B:
  - max_rolling_avg: rolling window of the target within the same event.
    MITIGATED: dropped at Step 2 of pipeline (before any computation).
  - evDur_Hr, evDur_day: post-event quantities. MITIGATED: dropped.
  - median_sum: unknown lineage aggregate. MITIGATED: dropped.
  - Imputation medians: computed on full dataset — mild imputation leakage
    if used without proper cross-validation. MUST refit medians on train fold only.

Residual risks:
  - year trend in Dataset B: year=2022 may "explain" higher severity not through
    physical causation but data expansion. Monitor year feature importance.
''')

print('''
5. RECOMMENDED FEATURES FOR THE ML MODEL
---
Model A — Transformer Risk Classifier:
  Input features (20 columns, keep all):
    Raw sensors (14): hydrogen_ppm, oxygen_ppm, nitrogen_ppm, methane_ppm,
      co_ppm, co2_ppm, ethylene_ppm, ethane_ppm, acetylene_ppm, dbds_mg_kg,
      power_factor_pct, interfacial_tension_mNm, dielectric_kv, water_content_ppm
    Engineered (6): tdcg_ppm, rogers_r1_ch4_h2, rogers_r2_c2h2_c2h4,
      rogers_r3_c2h2_ch4, co2_co_ratio, ethylene_ethane_ratio
  Target: risk_int (0=LOW, 1=MEDIUM, 2=HIGH)
  Algorithm: RandomForestClassifier with class_weight="balanced"
  Note: Use log1p transform + StandardScaler for distance-based baselines.

Model B — Weather Outage Severity Classifier:
  Weather features (4): tmpf, relh, sknt, p01i
  Temporal features (4): hour_of_day, month, year, season (encode as dummies)
  Location feature (1): state (label or target-encode; drop county + fips)
  Total: ~15-20 features after encoding
  Target: outage_severity_int (0=LOW, 1=MEDIUM, 2=HIGH)
  Algorithm: RandomForestClassifier with class_weight="balanced"
  Caution: HIGH class has only 456 samples (1.4%) — consider binary framing
    (LOW vs MEDIUM+HIGH) as an alternative to 3-class.

Scoring fusion (inference):
  combined_risk = model_A_risk_proba * model_B_severity_proba * criticality_weight
  -> rank assets by combined_risk descending for maintenance prioritisation
''')